# 03 — The Logit Lens

At every layer, *if the model had to answer now, what would it say?*
The logit lens projects each layer's residual stream through the final
LayerNorm + unembedding, revealing at which depth each prediction forms
(nostalgebraist, 2020).

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

In [ ]:
# Train a tiny model on a synthetic corpus (~30s on CPU).
# The corpus is a seeded word-salad: repetitive enough to learn, varied enough
# that BPE cannot collapse it into a handful of giant tokens.
import random

from kamui.tokenizer.bpe import BPETokenizer
from kamui.training import DataLoader, TextDataset, Trainer, TrainingConfig

rng = random.Random(0)
WORDS = ["the", "cat", "dog", "sat", "ran", "on", "to", "mat", "log", "sun"]
CORPUS = " ".join(rng.choice(WORDS) for _ in range(4000))

config = ModelConfig(n_layers=2, d_model=64, n_heads=4, d_ff=128,
                     vocab_size=300, context_length=32, dropout=0.0)
tokenizer = BPETokenizer.train(CORPUS, vocab_size=config.vocab_size)
tokens = tokenizer.encode(CORPUS)

model = kamui.KAMUITransformer(config)
trainer = Trainer(
    model,
    DataLoader(TextDataset(tokens, config.context_length), batch_size=8, seed=0),
    config=TrainingConfig(max_lr=3e-3, warmup_steps=10, max_steps=1000),
)
records = trainer.train(150)
model.eval()
print(f"loss: {records[0]['train_loss']:.3f} -> {records[-1]['train_loss']:.3f}")

In [ ]:
from kamui.mechinterp import LogitLens

ids = torch.tensor(tokenizer.encode("the cat sat on the"))
result = LogitLens(model, tokenizer).run(ids)
print("probs shape (layers+1, S, V):", tuple(result.probs.shape))
result.plot()

In [ ]:
# Watch the final prediction sharpen with depth at the last position.
result.plot_position(len(ids) - 1)